In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv
/kaggle/input/competitions/home-data-for-ml-course/sample_submission.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/data_description.txt
/kaggle/input/competitions/home-data-for-ml-course/test.csv.gz
/kaggle/input/competitions/home-data-for-ml-course/train.csv
/kaggle/input/competitions/home-data-for-ml-course/test.csv


In [2]:
# =========================================================
# House Prices Prediction using XGBoost
# =========================================================

# Numerical operations
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Cross validation and optimization
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

# Pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Imputation methods
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

# Enable IterativeImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Encoders
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# XGBoost model
from xgboost import XGBRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# =========================================================
# Dataset Loading
# =========================================================

# Read training dataset
housing = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/train.csv"
)

# Read test dataset
test_data = pd.read_csv(
    "/kaggle/input/competitions/home-data-for-ml-course/test.csv"
)

# Display first 5 rows
housing.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
# =========================================================
# Dataset Inspection
# =========================================================

print("Training Set Shape:")
print(housing.shape)

print("\nTest Set Shape:")
print(test_data.shape)

print("\nColumn Names:")
print(housing.columns)

print("\nMissing Values:")
print(housing.isnull().sum().sort_values(ascending=False).head(10))

Training Set Shape:
(1460, 81)

Test Set Shape:
(1459, 80)

Column Names:
Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 

In [5]:
# =========================================================
# Q1(a)
# Numerical Features + SimpleImputer + XGBoost
# =========================================================

# Separate target variable
y = housing["SalePrice"]

# Remove target column from feature set
X = housing.drop("SalePrice", axis=1)

# Extract numerical feature names
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

# Keep only numerical features
X_num = X[num_cols]

# Create machine learning pipeline
pipeline_simple = Pipeline([
    
    # Missing value imputation
    ("imputer", SimpleImputer(
        strategy="median"
    )),
    
    # XGBoost regression model
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

# Perform 5-fold cross validation
scores = cross_val_score(
    pipeline_simple,
    X_num,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

# Convert negative RMSE values to positive
rmse_scores = -scores

# Print results
print("Fold RMSE Scores:")
print(rmse_scores)

print("\nAverage RMSE:")
print(rmse_scores.mean())

Fold RMSE Scores:
[28682.22265625 35734.80859375 30874.15820312 22176.09570312
 32647.55664062]

Average RMSE:
30022.968359375


### Q1(a) Result

In this experiment, only numerical features were used as input variables. Missing values were handled using SimpleImputer with the median strategy, and an XGBoost regressor with `n_estimators=100` and `random_state=0` was applied within a machine learning pipeline. 

A 5-fold cross-validation experiment was performed on the training dataset using the negative root mean squared error metric. The obtained RMSE scores for each fold are shown below:

- Fold 1 RMSE: 28682.22
- Fold 2 RMSE: 35734.81
- Fold 3 RMSE: 30874.16
- Fold 4 RMSE: 22176.10
- Fold 5 RMSE: 32647.56

The average RMSE obtained from cross-validation is:

```text
30022.97

In [6]:
# =========================================================
# Q1(b)
# SimpleImputer with Missing Indicators
# =========================================================

pipeline_indicator = Pipeline([
    
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

scores_indicator = cross_val_score(
    pipeline_indicator,
    X_num,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

rmse_indicator = -scores_indicator

print("Fold RMSE Scores:")
print(rmse_indicator)

print("\nAverage RMSE:")
print(rmse_indicator.mean())

Fold RMSE Scores:
[28084.4765625  35957.78515625 31075.70898438 21646.96679688
 32614.37109375]

Average RMSE:
29875.86171875


### Q1(b) Result

In this experiment, missing value indicator columns were added by setting `add_indicator=True` in the `SimpleImputer`. A 5-fold cross-validation experiment was performed using numerical features only together with an XGBoost regressor.

The obtained RMSE scores for each fold are:

- Fold 1 RMSE: 28084.48
- Fold 2 RMSE: 35957.79
- Fold 3 RMSE: 31075.71
- Fold 4 RMSE: 21646.97
- Fold 5 RMSE: 32614.37

The average RMSE obtained from cross-validation is:

```text
29875.86

In [7]:
# =========================================================
# Q1(c)
# KNN Imputer
# =========================================================

pipeline_knn = Pipeline([
    
    ("imputer", KNNImputer(
        n_neighbors=7
    )),
    
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

scores_knn = cross_val_score(
    pipeline_knn,
    X_num,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

rmse_knn = -scores_knn

print("Fold RMSE Scores:")
print(rmse_knn)

print("\nAverage RMSE:")
print(rmse_knn.mean())

Fold RMSE Scores:
[27614.73242188 34972.02734375 31437.54492188 21864.90039062
 31984.3125    ]

Average RMSE:
29574.703515625


### Q1(c) Result

In this experiment, the `SimpleImputer` used in part (a) was replaced with `KNNImputer` by setting the `n_neighbors` parameter to 7. The KNN imputation method estimates missing values using the nearest neighboring samples in the dataset. Numerical features only were used together with an XGBoost regressor configured with `n_estimators=100` and `random_state=0`.

A 5-fold cross-validation experiment was performed using the negative root mean squared error metric. The obtained RMSE scores for each fold are shown below:

- Fold 1 RMSE: 27614.73
- Fold 2 RMSE: 34972.03
- Fold 3 RMSE: 31437.54
- Fold 4 RMSE: 21864.90
- Fold 5 RMSE: 31984.31

The average RMSE obtained from cross-validation is:

```text
29574.70

In [8]:
# =========================================================
# Q1(d)
# Iterative Imputer
# =========================================================

pipeline_iterative = Pipeline([
    
    ("imputer", IterativeImputer(
        random_state=0,
        max_iter=10,
        initial_strategy="median",
        sample_posterior=True
    )),
    
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

scores_iterative = cross_val_score(
    pipeline_iterative,
    X_num,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

rmse_iterative = -scores_iterative

print("Fold RMSE Scores:")
print(rmse_iterative)

print("\nAverage RMSE:")
print(rmse_iterative.mean())

Fold RMSE Scores:
[27269.40820312 35233.19140625 33291.49609375 21975.93945312
 28830.02148438]

Average RMSE:
29320.011328125


### Q1(d) Result

In this experiment, the `SimpleImputer` used in part (a) was replaced with `IterativeImputer`. The imputer was configured with `random_state=0`, `max_iter=10`, `initial_strategy="median"`, and `sample_posterior=True`. Numerical features only were used together with an XGBoost regressor having `n_estimators=100` and `random_state=0`.

A 5-fold cross-validation experiment was performed using the negative root mean squared error metric. The obtained RMSE scores for each fold are presented below:

- Fold 1 RMSE: 27269.41
- Fold 2 RMSE: 35233.19
- Fold 3 RMSE: 33291.50
- Fold 4 RMSE: 21975.94
- Fold 5 RMSE: 28830.02

The average RMSE obtained from cross-validation is:

```text
29320.01

### Q1(e) Result

Among all evaluated imputation methods, `IterativeImputer` achieved the best performance with the lowest average RMSE score of approximately:

29320.01

The overall RMSE comparison of the tested methods is summarized below:

* SimpleImputer: 30022.97
* SimpleImputer + Missing Indicators: 29875.86
* KNNImputer: 29574.70
* IterativeImputer: 29320.01

The superior performance of IterativeImputer may be explained by its ability to model relationships between features while estimating missing values. Unlike SimpleImputer, which replaces missing values using only a fixed statistical value such as the median, IterativeImputer predicts missing values by considering information from other related variables in the dataset.

Additionally, IterativeImputer can capture more complex feature interactions compared to KNNImputer, which relies mainly on neighboring samples. Since housing data contains strong correlations between variables such as house size, garage area, number of rooms, and sale price, modeling these relationships during imputation likely improved the quality of the generated values and reduced prediction error.


In [9]:
# =========================================================
# Q2(a)
# Full Preprocessing Pipeline with OneHotEncoder
# =========================================================

# Extract numerical columns
num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns

# Extract categorical columns
cat_cols = X.select_dtypes(
    include=["object"]
).columns

# Numerical preprocessing pipeline
# Best method from Q1: IterativeImputer
num_transformer = Pipeline([
    
    ("imputer", IterativeImputer(
        random_state=0,
        max_iter=10,
        initial_strategy="median",
        sample_posterior=True
    ))
])

# Categorical preprocessing pipeline
# First fill missing categorical values, then apply one-hot encoding
cat_transformer = Pipeline([
    
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols)
])

# Full machine learning pipeline
full_pipeline = Pipeline([
    
    ("preprocessor", preprocessor),
    
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

# Perform cross validation
scores_full = cross_val_score(
    full_pipeline,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

# Convert negative RMSE to positive
rmse_full = -scores_full

# Print results
print("Fold RMSE Scores:")
print(rmse_full)

print("\nAverage RMSE:")
print(rmse_full.mean())

if rmse_full.mean() < rmse_iterative.mean():
    print("\nIncluding categorical features improved the performance compared to the best numerical-only model.")
else:
    print("\nIncluding categorical features did not improve the performance compared to the best numerical-only model.")

Fold RMSE Scores:
[27510.12890625 31608.07617188 30350.63085938 21939.16015625
 31016.13867188]

Average RMSE:
28484.826953125

Including categorical features improved the performance compared to the best numerical-only model.


### Q2(a) Result

In this experiment, a full preprocessing pipeline was created using `ColumnTransformer` and `Pipeline`. The best imputation method from Question 1, `IterativeImputer`, was applied to numerical features. For categorical features, missing values were filled using the most frequent value, and `OneHotEncoder` was applied with `handle_unknown="ignore"`.

The obtained 5-fold cross-validation RMSE scores are:

- Fold 1 RMSE: 27510.13
- Fold 2 RMSE: 31608.08
- Fold 3 RMSE: 30350.63
- Fold 4 RMSE: 21939.16
- Fold 5 RMSE: 31016.14

The average RMSE is:

```text
28484.83

In [10]:
# =========================================================
# Q2(b)
# Ordinal Encoding
# =========================================================

# Categorical preprocessing pipeline
# First fill missing categorical values, then apply ordinal encoding
cat_transformer_ord = Pipeline([
    
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan
    ))
])

# Full preprocessing
preprocessor_ord = ColumnTransformer([
    
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer_ord, cat_cols)
])

# Full pipeline
pipeline_ord = Pipeline([
    
    ("preprocessor", preprocessor_ord),
    
    ("model", XGBRegressor(
        random_state=0,
        n_estimators=100
    ))
])

# Cross validation
scores_ord = cross_val_score(
    pipeline_ord,
    X,
    y,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

# Convert scores
rmse_ord = -scores_ord

# Print results
print("Fold RMSE Scores:")
print(rmse_ord)

print("\nAverage RMSE:")
print(rmse_ord.mean())

if rmse_ord.mean() < rmse_full.mean():
    print("\nOrdinal encoding performed better than one-hot encoding.")
else:
    print("\nOne-hot encoding performed better than ordinal encoding.")

Fold RMSE Scores:
[25271.93164062 32891.53515625 30917.59179688 22392.06054688
 25765.24414062]

Average RMSE:
27447.67265625

Ordinal encoding performed better than one-hot encoding.


### Q2(b) Result

In this experiment, one-hot encoding was replaced with ordinal encoding for categorical features. Missing categorical values were filled using the most frequent value, and `OrdinalEncoder` was applied with `handle_unknown="use_encoded_value"` and `unknown_value=np.nan`.

The obtained 5-fold cross-validation RMSE scores are:

- Fold 1 RMSE: 25271.93
- Fold 2 RMSE: 32891.54
- Fold 3 RMSE: 30917.59
- Fold 4 RMSE: 22392.06
- Fold 5 RMSE: 25765.24

The average RMSE is:

```text
27447.67

- Ordinal encoding performed better than one-hot encoding because it achieved a lower average RMSE.

In [11]:
# =========================================================
# Q3
# Hyperparameter Optimization with GridSearchCV
# =========================================================

# Select the best preprocessing configuration from Q2
if rmse_full.mean() <= rmse_ord.mean():
    best_pipeline = full_pipeline
    best_encoding_name = "OneHotEncoder"
    best_q2_rmse = rmse_full.mean()
else:
    best_pipeline = pipeline_ord
    best_encoding_name = "OrdinalEncoder"
    best_q2_rmse = rmse_ord.mean()

print("Best preprocessing configuration from Q2:")
print(best_encoding_name)

# Parameter grid
param_grid = {
    
    "model__n_estimators": [100, 200],
    
    "model__max_depth": [3, 5],
    
    "model__learning_rate": [0.05, 0.1]
}

# Grid Search
grid_search = GridSearchCV(
    
    estimator=best_pipeline,
    
    param_grid=param_grid,
    
    cv=5,
    
    scoring="neg_root_mean_squared_error",
    
    n_jobs=-1,
    
    verbose=1
)

# Train all combinations
grid_search.fit(X, y)

# Best parameters
best_params = grid_search.best_params_

# Best RMSE
best_rmse = -grid_search.best_score_

print("\nBest Parameters:")
print(best_params)

print("\nBest RMSE:")
print(best_rmse)

if best_rmse < best_q2_rmse:
    print("\nHyperparameter optimization improved the performance.")
else:
    print("\nHyperparameter optimization did not improve the performance.")

Best preprocessing configuration from Q2:
OrdinalEncoder
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best Parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 5, 'model__n_estimators': 200}

Best RMSE:
27058.734375

Hyperparameter optimization improved the performance.


### Q3 Result

In this experiment, the best preprocessing configuration from Question 2 was used. The best configuration was `OrdinalEncoder`.

GridSearchCV was applied with 5-fold cross-validation to optimize the XGBoost hyperparameters `n_estimators`, `max_depth`, and `learning_rate`.

The best hyperparameter values are:

```text
learning_rate = 0.05
max_depth = 5
n_estimators = 200

The best RMSE score obtained from GridSearchCV is:

27058.73

Compared to the best result from Question 2, hyperparameter optimization improved the model performance because the RMSE decreased from 27447.67 to 27058.73.

In [12]:
# =========================================================
# Q4
# Final Prediction and Kaggle Submission
# =========================================================

# Generate predictions on test set using the best model from GridSearchCV
test_predictions = grid_search.predict(test_data)

# Create submission dataframe
submission = pd.DataFrame({
    
    "Id": test_data["Id"],
    
    "SalePrice": test_predictions
})

# Save submission file
submission.to_csv(
    "submission.csv",
    index=False
)

print("Submission file created successfully!")

submission.head()

Submission file created successfully!


,Id,SalePrice
0,1461,125372.687500
1,1462,148305.828125
2,1463,177785.281250
3,1464,188723.359375
4,1465,191572.265625


In [13]:
# =========================================================
# Final Summary
# =========================================================

results = pd.DataFrame({
    
    "Method": [
        "SimpleImputer",
        "Indicator Features",
        "KNNImputer",
        "IterativeImputer",
        "OneHotEncoding",
        "OrdinalEncoding",
        "GridSearchCV Best Model"
    ],
    
    "RMSE": [
        rmse_scores.mean(),
        rmse_indicator.mean(),
        rmse_knn.mean(),
        rmse_iterative.mean(),
        rmse_full.mean(),
        rmse_ord.mean(),
        best_rmse
    ]
})

results = results.sort_values(
    by="RMSE"
)

print(results)

                    Method          RMSE
6  GridSearchCV Best Model  27058.734375
5          OrdinalEncoding  27447.672656
4           OneHotEncoding  28484.826953
3         IterativeImputer  29320.011328
2               KNNImputer  29574.703516
1       Indicator Features  29875.861719
0            SimpleImputer  30022.968359


### Q4 Result

Using the best preprocessing configuration and the best hyperparameter values obtained from GridSearchCV, predictions were generated on the test dataset (`test.csv`). The final model used `OrdinalEncoder` as the best preprocessing configuration and the following optimized XGBoost hyperparameters:

```text
learning_rate = 0.05
max_depth = 5
n_estimators = 200

The best cross-validation RMSE score obtained during GridSearchCV was:

27058.73

After generating predictions on the test set, a submission file named submission.csv was created successfully. The file contained the required Id and SalePrice columns and was submitted to the Kaggle leaderboard.

The leaderboard score obtained on the Kaggle test set is:

14954.67344

The leaderboard ranking is:

544

The final submission was successfully evaluated on the Kaggle leaderboard. Although the most recent submission score was not an improvement over the previous best score, the optimized XGBoost model produced a valid prediction file for the test dataset.